# Ensemble Learning Methods: Classifying Music Genres

In this tutorial, we'll explore **ensemble learning** — one of the most powerful families of machine learning techniques — by building models that classify music tracks into genres based on audio features.

Imagine you have a dataset of tracks from a streaming service like Spotify, and each track comes with numerical descriptors: how fast it is, how loud, how danceable, etc. Can we predict the genre from these features alone?

We'll start with a simple decision tree, then progressively build more sophisticated ensemble models — **Bagging**, **Random Forests**, **AdaBoost**, and **Gradient Boosting** — and see how combining many weak learners produces dramatically better results.

**What you'll learn:**
- Why single decision trees tend to overfit and how ensembles fix that
- The difference between *bagging* (parallel ensembles) and *boosting* (sequential ensembles)
- How to visualize and interpret feature importances
- How to tune hyperparameters and compare models rigorously with cross-validation

---
## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import warnings
warnings.filterwarnings("ignore")

# Plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All imports loaded successfully.")

---
## 2. Simulating a Music Track Dataset

We'll create a synthetic dataset that mimics what you'd find in a real Spotify-style audio features API. Each track has nine numerical features and belongs to one of four genres: **Rock**, **Pop**, **Hip-Hop**, or **Classical**.

The feature distributions are designed so that genres have *overlapping but distinguishable* characteristics — just like real music data.

In [ ]:
def generate_music_dataset(n_per_genre=250, seed=42):
    """Generate a synthetic music dataset with realistic feature distributions per genre."""
    rng = np.random.RandomState(seed)
    records = []

    # --- Genre profiles (mean, std) for each feature ---
    # Features: tempo, energy, danceability, loudness, speechiness,
    #           acousticness, instrumentalness, valence, duration_ms
    profiles = {
        "Rock": {
            "tempo": (128, 18),
            "energy": (0.75, 0.12),
            "danceability": (0.50, 0.13),
            "loudness": (-6.0, 2.5),
            "speechiness": (0.05, 0.03),
            "acousticness": (0.15, 0.10),
            "instrumentalness": (0.20, 0.18),
            "valence": (0.50, 0.18),
            "duration_ms": (240000, 50000),
        },
        "Pop": {
            "tempo": (118, 20),
            "energy": (0.68, 0.14),
            "danceability": (0.72, 0.10),
            "loudness": (-5.5, 2.0),
            "speechiness": (0.08, 0.05),
            "acousticness": (0.22, 0.15),
            "instrumentalness": (0.02, 0.03),
            "valence": (0.65, 0.16),
            "duration_ms": (210000, 30000),
        },
        "Hip-Hop": {
            "tempo": (95, 22),
            "energy": (0.65, 0.15),
            "danceability": (0.78, 0.09),
            "loudness": (-7.0, 2.8),
            "speechiness": (0.22, 0.12),
            "acousticness": (0.12, 0.10),
            "instrumentalness": (0.01, 0.02),
            "valence": (0.55, 0.20),
            "duration_ms": (220000, 40000),
        },
        "Classical": {
            "tempo": (105, 30),
            "energy": (0.25, 0.15),
            "danceability": (0.28, 0.12),
            "loudness": (-18.0, 5.0),
            "speechiness": (0.04, 0.02),
            "acousticness": (0.88, 0.10),
            "instrumentalness": (0.82, 0.15),
            "valence": (0.30, 0.15),
            "duration_ms": (320000, 90000),
        },
    }

    for genre, feats in profiles.items():
        for _ in range(n_per_genre):
            row = {"genre": genre}
            for feat_name, (mu, sigma) in feats.items():
                val = rng.normal(mu, sigma)
                # Clip 0-1 features to valid range
                if feat_name in [
                    "energy", "danceability", "speechiness",
                    "acousticness", "instrumentalness", "valence",
                ]:
                    val = np.clip(val, 0.0, 1.0)
                if feat_name == "duration_ms":
                    val = max(val, 60000)  # at least 1 minute
                row[feat_name] = val
            records.append(row)

    df = pd.DataFrame(records).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df


df = generate_music_dataset(n_per_genre=250, seed=RANDOM_STATE)
print(f"Dataset shape: {df.shape}")
print(f"Genre distribution:\n{df['genre'].value_counts()}")
df.head(10)

In [ ]:
df.describe().round(3)

---
## 3. Exploratory Data Analysis

Before modeling, let's visualize how the features differ across genres. This gives us intuition about which features will be most useful for classification.

In [ ]:
feature_cols = [
    "tempo", "energy", "danceability", "loudness", "speechiness",
    "acousticness", "instrumentalness", "valence", "duration_ms",
]

genre_palette = {
    "Rock": "#e74c3c",
    "Pop": "#3498db",
    "Hip-Hop": "#2ecc71",
    "Classical": "#9b59b6",
}

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.ravel(), feature_cols):
    for genre in df["genre"].unique():
        subset = df[df["genre"] == genre]
        ax.hist(
            subset[col], bins=25, alpha=0.45,
            label=genre, color=genre_palette[genre],
        )
    ax.set_title(col, fontsize=13, fontweight="bold")
    ax.legend(fontsize=8)

fig.suptitle("Feature Distributions by Genre", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

A few things jump out right away:
- **Classical** music is very distinct — it's quiet (low loudness), highly acoustic, and highly instrumental.
- **Hip-Hop** has the highest speechiness (vocals/rapping) and highest danceability.
- **Rock** and **Pop** overlap quite a bit in energy and tempo, which will make them harder to separate.

Let's also look at pairwise feature relationships.

In [ ]:
# Scatter plot of two highly discriminative features
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for genre in df["genre"].unique():
    mask = df["genre"] == genre
    axes[0].scatter(
        df.loc[mask, "acousticness"], df.loc[mask, "energy"],
        alpha=0.5, label=genre, color=genre_palette[genre], s=25,
    )
axes[0].set_xlabel("Acousticness")
axes[0].set_ylabel("Energy")
axes[0].set_title("Energy vs. Acousticness")
axes[0].legend()

for genre in df["genre"].unique():
    mask = df["genre"] == genre
    axes[1].scatter(
        df.loc[mask, "speechiness"], df.loc[mask, "danceability"],
        alpha=0.5, label=genre, color=genre_palette[genre], s=25,
    )
axes[1].set_xlabel("Speechiness")
axes[1].set_ylabel("Danceability")
axes[1].set_title("Danceability vs. Speechiness")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Train-Test Split

We'll hold out 25% of the data for final testing, and use the training set for model fitting and cross-validation.

In [ ]:
X = df[feature_cols].copy()
y = df["genre"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nClass balance (train):\n{y_train.value_counts()}")

---
## 5. Decision Tree Baseline

A single **decision tree** is the building block of every ensemble method we'll cover. It works by recursively splitting the data on feature thresholds that best separate the classes.

Decision trees are easy to interpret, but they have a critical flaw: they tend to **overfit**. A fully grown tree memorizes the training data and generalizes poorly. Let's see this in action.

In [ ]:
# Fit an unrestricted tree (likely to overfit)
dt_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_full.fit(X_train, y_train)

train_acc = dt_full.score(X_train, y_train)
test_acc = dt_full.score(X_test, y_test)

print(f"Unrestricted Decision Tree")
print(f"  Training accuracy: {train_acc:.4f}")
print(f"  Test accuracy:     {test_acc:.4f}")
print(f"  Tree depth:        {dt_full.get_depth()}")
print(f"  Number of leaves:  {dt_full.get_n_leaves()}")
print(f"\nNotice the gap between train and test — that's overfitting.")

In [ ]:
# Now fit a pruned tree (max_depth=4) — more generalizable
dt_pruned = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
dt_pruned.fit(X_train, y_train)

train_acc_p = dt_pruned.score(X_train, y_train)
test_acc_p = dt_pruned.score(X_test, y_test)

print(f"Pruned Decision Tree (max_depth=4)")
print(f"  Training accuracy: {train_acc_p:.4f}")
print(f"  Test accuracy:     {test_acc_p:.4f}")
print(f"  Tree depth:        {dt_pruned.get_depth()}")
print(f"  Number of leaves:  {dt_pruned.get_n_leaves()}")

### Visualizing the Decision Tree

One of the best things about decision trees is that you can actually *see* the logic. Let's visualize our pruned tree.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt_pruned,
    feature_names=feature_cols,
    class_names=sorted(y.unique()),
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    proportion=True,
)
ax.set_title("Pruned Decision Tree (max_depth=4)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

You can trace any path from the root to a leaf and read it as a set of if-then rules. For example, you might see something like: *"If acousticness > 0.55 and instrumentalness > 0.45, predict Classical."*

The colors indicate the dominant class at each node, and the intensity reflects confidence.

In [ ]:
# Detailed classification report for the pruned tree
y_pred_dt = dt_pruned.predict(X_test)
print("Classification Report — Pruned Decision Tree\n")
print(classification_report(y_test, y_pred_dt))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dt, ax=ax, cmap="Blues",
    display_labels=sorted(y.unique()),
)
ax.set_title("Confusion Matrix — Decision Tree (depth=4)", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Bagging — Bootstrap Aggregation

The core idea of **bagging** is simple but powerful:

1. Create many **bootstrap samples** (random samples drawn *with replacement*) from the training data.
2. Train an independent decision tree on each bootstrap sample.
3. Combine their predictions by **majority vote**.

Each individual tree still overfits, but because they each see a slightly different version of the data, their errors are *different*. When you average them out, the overfitting cancels and you get a more stable, accurate model.

This is the **variance reduction** principle: bagging reduces variance without increasing bias.

In [ ]:
# BaggingClassifier wraps any base estimator with bootstrap sampling
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=100,
    max_samples=0.8,       # each tree sees 80% of the data
    bootstrap=True,        # sample WITH replacement
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
bag_clf.fit(X_train, y_train)

train_acc_bag = bag_clf.score(X_train, y_train)
test_acc_bag = bag_clf.score(X_test, y_test)

print(f"Bagging Classifier (100 trees)")
print(f"  Training accuracy: {train_acc_bag:.4f}")
print(f"  Test accuracy:     {test_acc_bag:.4f}")
print(f"\nCompare this to the single tree test accuracy of {test_acc:.4f} (unpruned) / {test_acc_p:.4f} (pruned).")
print(f"Bagging reduces the gap between train and test accuracy — less overfitting!")

In [ ]:
# Let's see how performance changes as we add more trees
n_tree_range = [1, 5, 10, 25, 50, 75, 100, 150, 200]
bag_scores = []

for n in n_tree_range:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=n, bootstrap=True,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    bag.fit(X_train, y_train)
    bag_scores.append(bag.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(n_tree_range, bag_scores, "o-", color="#e67e22", linewidth=2, markersize=7)
plt.xlabel("Number of Trees")
plt.ylabel("Test Accuracy")
plt.title("Bagging: Test Accuracy vs. Number of Trees", fontweight="bold")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice how accuracy improves quickly at first, then plateaus.")
print("Adding more trees never hurts — it just stops helping after a point.")

---
## 7. Random Forest

A **Random Forest** is bagging with one extra twist: at each split in each tree, the algorithm only considers a *random subset* of features (typically $\sqrt{p}$ features out of $p$ total).

Why does this help? In plain bagging, if one feature is extremely strong, every tree will split on it first, making all the trees look similar. By forcing trees to consider different subsets of features, Random Forests produce more **diverse** trees — and diversity is what makes ensembles powerful.

Think of it like a committee of experts: you want each expert to have a different perspective, not for everyone to parrot the same opinion.

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,          # let trees grow fully
    max_features="sqrt",     # random subset of features at each split
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_clf.fit(X_train, y_train)

train_acc_rf = rf_clf.score(X_train, y_train)
test_acc_rf = rf_clf.score(X_test, y_test)

print(f"Random Forest (100 trees)")
print(f"  Training accuracy: {train_acc_rf:.4f}")
print(f"  Test accuracy:     {test_acc_rf:.4f}")

# Cross-validation gives us a more robust estimate
cv_scores_rf = cross_val_score(rf_clf, X_train, y_train, cv=5, scoring="accuracy")
print(f"\n  5-fold CV accuracy: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})")

In [ ]:
y_pred_rf = rf_clf.predict(X_test)
print("Classification Report — Random Forest\n")
print(classification_report(y_test, y_pred_rf))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, ax=ax, cmap="Greens",
    display_labels=sorted(y.unique()),
)
ax.set_title("Confusion Matrix — Random Forest", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 8. AdaBoost — Adaptive Boosting

Bagging and Random Forests build trees **in parallel** — each tree is independent. **Boosting** takes the opposite approach: trees are built **sequentially**, where each new tree focuses on the mistakes of the previous ones.

Here's the AdaBoost algorithm in plain English:

1. Train a weak classifier (typically a shallow decision tree, called a "stump").
2. Look at which training samples it got *wrong*.
3. **Increase the weight** of those misclassified samples so the next tree pays more attention to them.
4. Train the next tree on the re-weighted data.
5. Repeat, and combine all trees with a weighted vote.

The key insight: boosting reduces **bias** (underfitting), while bagging reduces **variance** (overfitting). They solve different problems.

In [ ]:
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # stumps (depth-1 trees)
    n_estimators=100,
    learning_rate=0.5,
    random_state=RANDOM_STATE,
    algorithm="SAMME",
)
ada_clf.fit(X_train, y_train)

train_acc_ada = ada_clf.score(X_train, y_train)
test_acc_ada = ada_clf.score(X_test, y_test)

print(f"AdaBoost (100 stumps, lr=0.5)")
print(f"  Training accuracy: {train_acc_ada:.4f}")
print(f"  Test accuracy:     {test_acc_ada:.4f}")

cv_scores_ada = cross_val_score(ada_clf, X_train, y_train, cv=5, scoring="accuracy")
print(f"\n  5-fold CV accuracy: {cv_scores_ada.mean():.4f} (+/- {cv_scores_ada.std():.4f})")

In [ ]:
# Visualize how AdaBoost improves as more stumps are added
ada_staged_scores = []
for y_staged_pred in ada_clf.staged_predict(X_test):
    ada_staged_scores.append(accuracy_score(y_test, y_staged_pred))

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(ada_staged_scores) + 1), ada_staged_scores,
         color="#c0392b", linewidth=2)
plt.xlabel("Number of Boosting Rounds")
plt.ylabel("Test Accuracy")
plt.title("AdaBoost: Accuracy Improves with Each Boosting Round", fontweight="bold")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Each new stump corrects the errors of the previous ensemble.")
print("Early rounds make the biggest jumps; later rounds fine-tune.")

---
## 9. Gradient Boosting — A Brief Introduction

**Gradient Boosting** is another sequential ensemble method, but instead of re-weighting samples (like AdaBoost), it fits each new tree to the **residual errors** (gradients of the loss function) of the current ensemble.

Think of it this way:
- The first tree makes predictions.
- The second tree tries to predict *what the first tree got wrong* (the residuals).
- The third tree predicts the residuals of the first two combined.
- And so on.

Each tree is a small correction to the previous ensemble. The `learning_rate` parameter controls how much each tree's correction counts — smaller values require more trees but tend to generalize better.

Gradient Boosting is one of the most successful algorithms in practice, powering winners of many Kaggle competitions (via libraries like XGBoost and LightGBM).

In [ ]:
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,          # stochastic gradient boosting (use 80% of data per tree)
    random_state=RANDOM_STATE,
)
gb_clf.fit(X_train, y_train)

train_acc_gb = gb_clf.score(X_train, y_train)
test_acc_gb = gb_clf.score(X_test, y_test)

print(f"Gradient Boosting (100 trees, depth=3, lr=0.1)")
print(f"  Training accuracy: {train_acc_gb:.4f}")
print(f"  Test accuracy:     {test_acc_gb:.4f}")

cv_scores_gb = cross_val_score(gb_clf, X_train, y_train, cv=5, scoring="accuracy")
print(f"\n  5-fold CV accuracy: {cv_scores_gb.mean():.4f} (+/- {cv_scores_gb.std():.4f})")

In [ ]:
# Staged performance for Gradient Boosting
gb_staged_train = []
gb_staged_test = []
for i, y_staged_pred in enumerate(gb_clf.staged_predict(X_test)):
    gb_staged_test.append(accuracy_score(y_test, y_staged_pred))
for i, y_staged_pred in enumerate(gb_clf.staged_predict(X_train)):
    gb_staged_train.append(accuracy_score(y_train, y_staged_pred))

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(gb_staged_train) + 1), gb_staged_train,
         label="Train", color="#2980b9", linewidth=2)
plt.plot(range(1, len(gb_staged_test) + 1), gb_staged_test,
         label="Test", color="#e67e22", linewidth=2)
plt.xlabel("Number of Boosting Rounds")
plt.ylabel("Accuracy")
plt.title("Gradient Boosting: Train vs. Test Accuracy Over Rounds", fontweight="bold")
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Watch for the gap between train and test curves — that tells you when overfitting begins.")

---
## 10. Feature Importance Comparison

One of the most practical outputs of tree-based ensembles is **feature importance** — a measure of how much each feature contributes to the model's predictions. In sklearn, this is computed from how much each feature reduces impurity (Gini or entropy) across all trees.

Let's compare what each model considers important.

In [ ]:
# Gather feature importances from each model
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Decision Tree": dt_pruned.feature_importances_,
    "Random Forest": rf_clf.feature_importances_,
    "AdaBoost": ada_clf.feature_importances_,
    "Gradient Boosting": gb_clf.feature_importances_,
})

importance_df = importance_df.set_index("Feature")
importance_df = importance_df.loc[
    importance_df["Random Forest"].sort_values(ascending=False).index
]

print("Feature Importances (sorted by Random Forest):\n")
print(importance_df.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
models_for_plot = [
    ("Decision Tree", "#e74c3c"),
    ("Random Forest", "#27ae60"),
    ("AdaBoost", "#8e44ad"),
    ("Gradient Boosting", "#2980b9"),
]

for ax, (model_name, color) in zip(axes.ravel(), models_for_plot):
    sorted_imp = importance_df[model_name].sort_values(ascending=True)
    ax.barh(sorted_imp.index, sorted_imp.values, color=color, alpha=0.8)
    ax.set_xlabel("Importance")
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_xlim(0, importance_df.values.max() * 1.1)

fig.suptitle("Feature Importances Across Models", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- All models agree that acousticness and instrumentalness are top features")
print("  (these separate Classical from everything else).")
print("- Speechiness is important for identifying Hip-Hop.")
print("- The single decision tree concentrates importance on fewer features,")
print("  while ensemble models spread importance more evenly — they use more information.")

---
## 11. Hyperparameter Tuning

Let's tune the Gradient Boosting model using `GridSearchCV`. The three most impactful hyperparameters are:

- **`n_estimators`**: number of boosting rounds (more = more capacity, but slower and risk of overfitting)
- **`max_depth`**: depth of each individual tree (deeper = more complex interactions captured)
- **`learning_rate`**: step size for each tree's contribution (smaller = more conservative, usually better with more trees)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 3, 5],
    "learning_rate": [0.05, 0.1, 0.2],
}

gb_search = GridSearchCV(
    GradientBoostingClassifier(subsample=0.8, random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)
gb_search.fit(X_train, y_train)

print(f"Best parameters: {gb_search.best_params_}")
print(f"Best CV accuracy: {gb_search.best_score_:.4f}")
print(f"\nTest accuracy with best model: {gb_search.score(X_test, y_test):.4f}")

In [ ]:
# Visualize how learning_rate and n_estimators interact
results = pd.DataFrame(gb_search.cv_results_)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, depth in zip(axes, [2, 3, 5]):
    for lr in [0.05, 0.1, 0.2]:
        mask = (results["param_max_depth"] == depth) & (results["param_learning_rate"] == lr)
        subset = results[mask].sort_values("param_n_estimators")
        ax.plot(
            subset["param_n_estimators"].astype(int),
            subset["mean_test_score"],
            "o-", label=f"lr={lr}", linewidth=2, markersize=6,
        )
    ax.set_xlabel("n_estimators")
    ax.set_ylabel("Mean CV Accuracy")
    ax.set_title(f"max_depth = {depth}", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Gradient Boosting: Hyperparameter Grid Search Results",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Takeaway: a lower learning rate with more trees often gives the best results,")
print("but there are diminishing returns. Deeper trees capture more complex patterns")
print("but are more prone to overfitting.")

---
## 12. Final Model Comparison

Let's put all our models side by side and compare them using cross-validated accuracy.

In [ ]:
# Collect cross-validated scores for all models
models = {
    "Decision Tree\n(depth=4)": dt_pruned,
    "Bagging\n(100 trees)": bag_clf,
    "Random Forest\n(100 trees)": rf_clf,
    "AdaBoost\n(100 stumps)": ada_clf,
    "Gradient Boosting\n(tuned)": gb_search.best_estimator_,
}

comparison_data = []

for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    test_acc = model.score(X_test, y_test)
    comparison_data.append({
        "Model": name,
        "CV Mean": cv.mean(),
        "CV Std": cv.std(),
        "Test Accuracy": test_acc,
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(12, 6))

colors = ["#e74c3c", "#e67e22", "#27ae60", "#8e44ad", "#2980b9"]
x_pos = range(len(comparison_df))

bars = ax.bar(
    x_pos,
    comparison_df["Test Accuracy"],
    color=colors,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
)

# Add error bars from CV
ax.errorbar(
    x_pos,
    comparison_df["CV Mean"],
    yerr=comparison_df["CV Std"],
    fmt="D", color="black", markersize=6,
    capsize=5, label="CV Mean +/- Std",
)

ax.set_xticks(x_pos)
ax.set_xticklabels(comparison_df["Model"], fontsize=11)
ax.set_ylabel("Accuracy", fontsize=13)
ax.set_title("Model Comparison: Music Genre Classification", fontsize=15, fontweight="bold")
ax.set_ylim(0.7, 1.0)
ax.legend(loc="lower right", fontsize=11)
ax.grid(axis="y", alpha=0.3)

# Annotate bars with values
for bar, val in zip(bars, comparison_df["Test Accuracy"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
        f"{val:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=11,
    )

plt.tight_layout()
plt.show()

---
## Wrap-Up: Key Takeaways

Here's what we covered and the core lessons:

### Decision Trees
- Great for interpretability; you can literally read the rules.
- But they overfit badly — an unrestricted tree memorizes the training data.

### Bagging (Bootstrap Aggregation)
- Trains many independent trees on bootstrapped samples and averages their votes.
- Reduces **variance** without increasing bias — a free improvement over single trees.

### Random Forests
- Bagging + random feature subsets at each split = more diverse trees.
- Usually outperforms plain bagging because the trees are less correlated.
- Excellent default choice — works well out of the box with minimal tuning.

### AdaBoost
- Sequential: each new tree focuses on previous mistakes by re-weighting samples.
- Reduces **bias** — turns many weak learners into one strong learner.
- Sensitive to noisy data and outliers (since it keeps increasing their weight).

### Gradient Boosting
- Also sequential, but fits trees to *residual errors* (loss gradients) instead of re-weighting.
- Often the top performer, especially with careful tuning of `learning_rate`, `n_estimators`, and `max_depth`.
- The `learning_rate` controls the trade-off between number of trees and per-tree contribution.

### Practical Advice
- **Start with Random Forest** — it's robust and fast with little tuning.
- **Graduate to Gradient Boosting** when you need maximum performance and are willing to tune.
- **Always use cross-validation** to evaluate models, not just a single train-test split.
- **Feature importances** from ensembles are a great tool for understanding your data, but remember they measure *predictive usefulness*, not causal effects.